# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [71]:
df = pd.read_csv('AviationData.csv', encoding='latin-1')

print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:\n{df.describe()}")

Shape: (88889, 31)

Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']

Data types:
Event.Id                   object
Investigation.Type         object
Accident.Number            object
Event.Date                 object
Location                   object
Country                    object
Latitude                   object
Longitude                  object
Airport.Code               object
Airport.Name               object
Injury.Severity            object
Aircraft.damage            object
Ai

C:\Users\Hunter\AppData\Local\Temp\ipykernel_31460\884928883.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('AviationData.csv', encoding='latin-1')


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [72]:
df = df[df['Aircraft.Category'] == 'Airplane']
df = df[df['Amateur.Built'] == 'No']
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'].dt.year >= 1983]

print(f"Final shape after filtering: {df.shape}")

Final shape after filtering: (21447, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [73]:
df['Total.Passengers'] = (df['Total.Fatal.Injuries'].fillna(0) + 
                          df['Total.Serious.Injuries'].fillna(0) + 
                          df['Total.Minor.Injuries'].fillna(0) + 
                          df['Total.Uninjured'].fillna(0))

df['Total.Passengers'] = df['Total.Passengers'].replace(0, np.nan)

df['Fatal.Serious.Injury.Fraction'] = (
    (df['Total.Fatal.Injuries'].fillna(0) + df['Total.Serious.Injuries'].fillna(0)) / 
    df['Total.Passengers']
)

df['Fatal.Serious.Injury.Fraction'] = df['Fatal.Serious.Injury.Fraction'].replace([np.inf, -np.inf], 0)
df['Fatal.Serious.Injury.Fraction'] = df['Fatal.Serious.Injury.Fraction'].fillna(0)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [74]:
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip()
df['Aircraft.Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(f"Destruction rate: {df['Aircraft.Destroyed'].mean():.2%}")

Destruction rate: 10.80%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [75]:
df['Make'] = df['Make'].str.strip()
df = df[df['Make'].notna()]

make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(f"Makes with 50+ occurrences: {len(valid_makes)}")
print(f"Shape after Make cleaning: {df.shape}")

Makes with 50+ occurrences: 42
Shape after Make cleaning: (17234, 34)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [76]:
df = df[df['Model'].notna()]
df['Model'] = df['Model'].str.strip()
df['Aircraft.Type'] = df['Make'] + ' ' + df['Model']

print(f"Shape after Model cleaning: {df.shape}")

Shape after Model cleaning: (17225, 35)


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [77]:
df['Engine.Type'] = df['Engine.Type'].str.strip()
df['Weather.Condition'] = df['Weather.Condition'].str.strip()
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip()

df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

print(f"Shape after column cleaning: {df.shape}")

Shape after column cleaning: (17225, 35)


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [78]:
non_null_counts = df.isnull().sum()
columns_to_keep = non_null_counts[non_null_counts <= len(df) - 20000].index

# If this removes all columns, use a less strict threshold
if len(columns_to_keep) == 0:
    columns_to_keep = non_null_counts[non_null_counts <= len(df) * 0.5].index

df = df[columns_to_keep]

print(f"Columns remaining: {len(df.columns)}")
print(f"Final shape: {df.shape}")

Columns remaining: 32
Final shape: (17225, 32)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [79]:
df.to_csv('aviation_cleaned.csv', index=False)

print("Cleaned data saved to 'aviation_cleaned.csv'")
print(f"Final dataset shape: {df.shape}")

Cleaned data saved to 'aviation_cleaned.csv'
Final dataset shape: (17225, 32)
